## Futásidő-összevetés: rprev referencia (naiv K-szoros) vs. kiterjesztett több indexdátumos implementáció

Ez a notebook a szakdolgozat 5. fejezetében rögzített több indexdátumos kiterjesztéshez kapcsolódó **futásidő-mérési protokollt** valósítja meg. A mérés célja annak kvantifikálása, hogy azonos bemeneti adaton és azonos szimulációs beállítások mellett milyen számítási igénye van  
(i) a referencia-megoldásnak, ahol a $\{t_k\}_{k=1}^K$ rácsot **$K$ külön `prevalence()` hívással** állítjuk elő, illetve  
(ii) a kiterjesztett rprev-implementációnak, ahol az `index` argumentum több időpontot ad meg, és a becslés **egyetlen eljárásfuttatásban** állítja elő a teljes prevalenciavektort és annak összefoglalóját.

### Kapcsolódás a dolgozathoz (5. fejezet)

A kiterjesztés matematikai tartalma röviden: adott $t_1 < \cdots < t_K$ rácson, az $r$-edik Monte Carlo futásban a futáson belül előállított incidens esetek $(D_{i\ell}^{(r)}, \mathbf{x}_{i\ell}^{(r)})$ és a futáson belül rögzített túlélési függvény $S^{(r)}(\cdot \mid \mathbf{x})$ alapján minden $t_k$ időpontra képezzük a prevalens státuszt, majd időpontonként összegezve kapjuk a $P^{(r)}(t_k)$ prevalenciakomponenseket és a futások feletti összefoglaló statisztikákat.

Csomagszinten a notebook azt a működést tekinti „kiterjesztett” megvalósításnak, ahol a `prevalence()` az `index` bemenetet rendezett `index_dates = (t_1,\ldots,t_K)` vektorrá normalizálja, a belső szimulációs réteg futásonként az `index_dates` rácson szolgáltat státuszinformációt, amelyből az összegző réteg időpontonként képez prevalencia-összefoglalót. A kimeneti objektum tartalmazza az `index_dates` vektort és az időpontindexelt eredményeket; a `print()`/`summary()` metódusok ezek szerint rendezve jelenítenek meg.

### Mérési egységek és protokoll

A dolgozat terminológiáját követve:

- **Alapkiértékelés:** a `prevalence()` egy futtatása rögzített $t$ mellett.
- **Eljárásfuttatás:** a $\{t_k\}_{k=1}^K$ időpontrács teljes kimenetének előállítása.

A referencia-megoldásban egy eljárásfuttatás **$K$ darab alapkiértékelésből** áll (külön-külön időpontra), míg a kiterjesztett megoldásban ugyanez **egyetlen alapkiértékelésként** valósul meg (általános esetben $K \ge 1$, speciális esetben $K=1$).

A notebook minden paraméterkombinációra ismételt futásidő-mérést végez (`reps`), és az összidőt `system.time(...)[["elapsed"]]` alapján rögzíti. A riportolt érték minden beállításpontra az ismétlések átlaga.

### Bemeneti adat és indexdátumok

A méréshez az rprev csomag `prevsim` példaadata szolgál alapul; ebből visszatevéses mintavétellel áll elő az $N$ soros, regiszterjellegű bemeneti tábla (a mérésben az $N$ a „rekordszámot” jelöli). Az indexdátumok egy rögzített kezdődátumtól (`index_start`) fix naplépéssel (`index_step_days`) generált, rendezett $K$ hosszú sorozatot alkotnak.

A notebookban az indexdátumok generálása determinisztikus: adott $K$, `index_start` és `index_step_days` mellett az `index_dates` teljesen reprodukálható, és a különböző mérési pontok között kizárólag a konfigurációs paraméterek változnak.

### Szimulációs és bootstrap beállítások

A mérések rácsa a következő fő paraméterek mentén szervezett:

- $N$: adatméret (rekordszám),
- $K$: indexdátumok száma,
- `N_boot`: bootstrap beállítás (túlélési görbék kezelése),

A notebook a `call_prevalence_safe()` segédfüggvénnyel gondoskodik arról, hogy a `prevalence()` csak a ténylegesen létező argumentumokat kapja meg. Ennek szerepe az, hogy a notebook csomagverziók között is futtatható maradjon, miközben az összehasonlítás logikája változatlan.

### Numerikus megjegyzés a több indexdátumos kiértékeléshez

A több indexdátumos kiértékelésben központi mennyiség az esetszintű diagnózisidő és az indexidőpont különbsége. Jelölje

$$
\Delta_{i\ell k} = t_k - D_{i\ell}.
$$

A $\Delta_{i\ell k} < 0$ esetek diagnózis előtti időpontot jelentenek, ahol prevalens státusz nem értelmezhető; ezt egy maszkkal kezeljük, például

$$
M_{i\ell k} = \mathbb{I}\{\Delta_{i\ell k} \ge 0\}.
$$

A kiértékelés ezek után a maszk szerinti pontokon végzi el a túlélési valószínűségek és a státuszindikátorok képzését, majd időpontonkénti aggregálással adja a prevalencia-idősor összetevőit.

### Kimenet

A futás végén egy eredménytábla készül, amely minden paraméterpontra tartalmazza:

- `N`, `K`, `N_boot`,
- `reps`,
- `elapsed_sec_mean`: az eljárásfuttatás átlagos falióra-ideje másodpercben.

A táblázat opcionálisan CSV-be is menthető, így a dolgozatban szereplő futásidő-ábrák és összegzések közvetlenül reprodukálhatók.

### Megjegyzés a dátumkezelésről

Az indexdátumok a hívás előtt egységesen `YYYY-MM-DD` formátumú karakterláncokká kerülnek alakításra. Ez a megoldás elkerüli azt a gyakorlati problémát, hogy ciklusváltozóként iterálva a `Date` osztályattribútum elveszhet, ami `NA`-vá parsolt indexdátumhoz és hibás futáshoz vezetne.

In [1]:
# ---- Közös benchmark beállítások (baseline + kiterjesztett összehasonlítás) ----
COMMON_CFG <- list(
  N_values              = c(1e2, 1e3),
  K_values              = c(1, 2, 3),
  N_boot_values         = c(10, 20),
  reps                  = 3L,
  num_years_to_estimate = c(3),
  index_start           = as.Date("2013-01-30"),
  index_step_days       = 30L,
  dist                  = "weibull",
  population_size       = 1e6
)


In [2]:
# =========================
# rprev pilot futásidő-mérés (baseline: K-szor prevalence())
# Kimenet: N, K, N_boot, reps, elapsed_sec_mean
# =========================

suppressPackageStartupMessages({
  library(rprev)
  library(survival)
})

# ---- Központi beállítások az előző cellából: COMMON_CFG ----

# ---- Alapadat ----
data(prevsim, package = "rprev")
base_df <- as.data.frame(prevsim)

# ---- Segédfüggvény: N soros "registry" a prevsim-ből ----
make_registry <- function(N, seed = 1L) {
  set.seed(as.integer(seed))
  base_df[sample.int(nrow(base_df), size = as.integer(N), replace = TRUE), , drop = FALSE]
}

# ---- Segédfüggvény: K db indexdátum (Date) ----
make_index_dates <- function(K, start_date, step_days) {
  seq.Date(from = start_date, by = as.integer(step_days), length.out = as.integer(K))
}

# ---- Biztonságos prevalence-hívás: csak a létező argumentumokat adja át ----
call_prevalence_safe <- function(...) {
  args <- list(...)
  fmls <- names(formals(rprev::prevalence))
  args <- args[names(args) %in% fmls]
  do.call(rprev::prevalence, args)
}

# ---- Egy futás: 1 indexdátumra ----
# index_date_chr: "YYYY-MM-DD"
run_rprev_once <- function(dat, index_date_chr, N_boot) {
  call_prevalence_safe(
    index = index_date_chr,
    num_years_to_estimate = COMMON_CFG$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = COMMON_CFG$dist,
    population_size = COMMON_CFG$population_size,
    death_column = "eventdate",
    N_boot = N_boot
  )
}

# ---- Mérés: K-szor hívjuk a prevalence()-t, és mérjük az összidőt ----
measure_one_setting <- function(N, K, N_boot, seed = 1L) {
  dat <- make_registry(N, seed = seed)

  idx_dates <- make_index_dates(K, COMMON_CFG$index_start, COMMON_CFG$index_step_days)

  # KRITIKUS FIX:
  # for(d in idx_dates) közben a Date class le tud esni -> előre karakterré alakítjuk.
  idx_dates_chr <- format(idx_dates, "%Y-%m-%d")

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  elapsed <- numeric(COMMON_CFG$reps)
  for (r in seq_len(COMMON_CFG$reps)) {
    gc()
    t <- system.time({
      for (d_chr in idx_dates_chr) {
        invisible(run_rprev_once(dat, d_chr, N_boot))
      }
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = COMMON_CFG$reps,
    elapsed_sec_mean = mean(elapsed)
  )
}

# ---- Grid futtatás ----
results <- do.call(
  rbind,
  lapply(COMMON_CFG$N_values, function(N) {
    do.call(rbind, lapply(COMMON_CFG$K_values, function(K) {
      do.call(rbind, lapply(COMMON_CFG$N_boot_values, function(N_boot) {
        message(sprintf("Run: N=%g, K=%d, N_boot=%d", N, K, N_boot))
          measure_one_setting(
            N = N,
            K = K,
            N_boot = N_boot,
            seed = 1000 + as.integer(N) + as.integer(K) + as.integer(N_boot)
          )
      }))
    }))
  })
)

results <- results[order(results$N, results$K, results$N_boot), ]
print(results)

# write.csv(results, "rprev_runtime_pilot.csv", row.names = FALSE)

Warning message:
"package 'rprev' was built under R version 4.4.3"
Run: N=100, K=1, N_boot=10

Run: N=100, K=1, N_boot=20

Run: N=100, K=2, N_boot=10

Run: N=100, K=2, N_boot=20

Run: N=100, K=3, N_boot=10

Run: N=100, K=3, N_boot=20

Run: N=1000, K=1, N_boot=10

Run: N=1000, K=1, N_boot=20

Run: N=1000, K=2, N_boot=10

Run: N=1000, K=2, N_boot=20

Run: N=1000, K=3, N_boot=10

Run: N=1000, K=3, N_boot=20



      N K N_boot reps elapsed_sec_mean
1   100 1     10    3       0.25000000
2   100 1     20    3       0.03000000
3   100 2     10    3       0.04333333
4   100 2     20    3       0.03000000
5   100 3     10    3       0.05333333
6   100 3     20    3       0.04666667
7  1000 1     10    3       0.02333333
8  1000 1     20    3       0.01666667
9  1000 2     10    3       0.04333333
10 1000 2     20    3       0.03666667
11 1000 3     10    3       0.06666667
12 1000 3     20    3       0.07000000


In [3]:
# Projekt gyökérkönyvtárának meghatározása
project_root <- trimws(system2("git", c("rev-parse", "--show-toplevel"), stdout = TRUE))
if (length(project_root) == 0 || !dir.exists(project_root)) {
  stop("Could not resolve project root from git.")
}

# A csomag lokális forráskódjának betöltése
if ("package:rprev" %in% search()) detach("package:rprev", unload = TRUE, character.only = TRUE)
if ("rprev" %in% loadedNamespaces()) unloadNamespace("rprev")
if (!requireNamespace("devtools", quietly = TRUE)) stop("devtools package is not available.")
suppressPackageStartupMessages(devtools::load_all(project_root, quiet = TRUE))
cat("Loaded rprev from: ", normalizePath(getNamespaceInfo("rprev", "path"), winslash = "/"), "\n", sep = "")

# Szintetikus, éves bontású regiszteradatot előállító szkript betöltése
sr_script <- file.path(project_root, "notebooks", "synthetic_registry_yearly.R")
if (file.exists(sr_script)) {
  source(sr_script)
} else {
  cat("Skipping source(): file not found -> ", sr_script, "\n", sep = "")
}


Warning message:
"package 'testthat' was built under R version 4.4.3"


Loaded rprev from: C:/Users/600972868/OneDrive - BT Plc/Desktop/Sajat Mappa/00 - BGE/Alkalmazott Matematika/Szakdolgozat/repo/rprev-ext
Skipping source(): file not found -> C:/Users/600972868/OneDrive - BT Plc/Desktop/Sajat Mappa/00 - BGE/Alkalmazott Matematika/Szakdolgozat/repo/rprev-ext/notebooks/synthetic_registry_yearly.R


In [4]:
# Git meta (branch + commit) kiírása
git_cmd <- function(args) {
  out <- tryCatch(system2("git", args, stdout = TRUE, stderr = FALSE), error = function(e) character(0))
  if (length(out) == 0) NA_character_ else out[[1]]
}

# Git ág és commit azonosító lekérdezése a projekt gyökerére hivatkozva
branch <- git_cmd(c("-C", project_root, "rev-parse", "--abbrev-ref", "HEAD"))
commit <- git_cmd(c("-C", project_root, "rev-parse", "HEAD"))

cat("Git branch: ", branch, "\n", sep = "")
cat("Git commit: ", commit, "\n", sep = "")


Warning message in system2("git", args, stdout = TRUE, stderr = FALSE):
"running command '"git" -C C:/Users/600972868/OneDrive - BT Plc/Desktop/Sajat Mappa/00 - BGE/Alkalmazott Matematika/Szakdolgozat/repo/rprev-ext rev-parse --abbrev-ref HEAD' had status 129"
Warning message in system2("git", args, stdout = TRUE, stderr = FALSE):
"running command '"git" -C C:/Users/600972868/OneDrive - BT Plc/Desktop/Sajat Mappa/00 - BGE/Alkalmazott Matematika/Szakdolgozat/repo/rprev-ext rev-parse HEAD' had status 129"


Git branch: NA
Git commit: NA


In [5]:
# ---- Kiterjesztett benchmark (natív multi-index hívás) ----
# Kimenet formátuma megegyezik a baseline táblával.

run_rprev_ext_once <- function(dat, index_dates_chr, N_boot) {
  call_prevalence_safe(
    index = index_dates_chr,
    num_years_to_estimate = COMMON_CFG$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = COMMON_CFG$dist,
    population_size = COMMON_CFG$population_size,
    death_column = "eventdate",
    N_boot = N_boot
  )
}

measure_one_setting_ext <- function(N, K, N_boot, seed = 1L) {
  dat <- make_registry(N, seed = seed)
  idx_dates <- make_index_dates(K, COMMON_CFG$index_start, COMMON_CFG$index_step_days)
  idx_dates_chr <- format(idx_dates, "%Y-%m-%d")

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  elapsed <- numeric(COMMON_CFG$reps)
  for (r in seq_len(COMMON_CFG$reps)) {
    gc()
    t <- system.time({
      invisible(run_rprev_ext_once(dat, idx_dates_chr, N_boot))
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = COMMON_CFG$reps,
    elapsed_sec_mean = mean(elapsed)
  )
}

results_ext <- do.call(
  rbind,
  lapply(COMMON_CFG$N_values, function(N) {
    do.call(rbind, lapply(COMMON_CFG$K_values, function(K) {
      do.call(rbind, lapply(COMMON_CFG$N_boot_values, function(N_boot) {
        message(sprintf("Ext run: N=%g, K=%d, N_boot=%d", N, K, N_boot))
          measure_one_setting_ext(
            N = N,
            K = K,
            N_boot = N_boot,
            seed = 2000 + as.integer(N) + as.integer(K) + as.integer(N_boot)
          )
      }))
    }))
  })
)

results_ext <- results_ext[order(results_ext$N, results_ext$K, results_ext$N_boot), ]
print(results_ext)


Ext run: N=100, K=1, N_boot=10

Ext run: N=100, K=1, N_boot=20

Ext run: N=100, K=2, N_boot=10

Ext run: N=100, K=2, N_boot=20

Ext run: N=100, K=3, N_boot=10

Ext run: N=100, K=3, N_boot=20

Ext run: N=1000, K=1, N_boot=10

Ext run: N=1000, K=1, N_boot=20

Ext run: N=1000, K=2, N_boot=10

Ext run: N=1000, K=2, N_boot=20

Ext run: N=1000, K=3, N_boot=10

Ext run: N=1000, K=3, N_boot=20



      N K N_boot reps elapsed_sec_mean
1   100 1     10    3       0.07000000
2   100 1     20    3       0.02333333
3   100 2     10    3       0.02000000
4   100 2     20    3       0.02666667
5   100 3     10    3       0.02666667
6   100 3     20    3       0.03000000
7  1000 1     10    3       0.02666667
8  1000 1     20    3       0.01666667
9  1000 2     10    3       0.03000000
10 1000 2     20    3       0.02333333
11 1000 3     10    3       0.03000000
12 1000 3     20    3       0.03333333
